In [2]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "12345678"
auth=(username, password)
driver = GraphDatabase.driver(uri, auth=(username, password))

In [3]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Neo4jMapReduce") \
    .master("local[*]") \
    .getOrCreate()

In [4]:
def Map(driver):

    query = """
    MATCH (c)
    WHERE c.kind = "Compound"
    OPTIONAL MATCH (c)-[r]->()
    RETURN c.name AS Compound, r.metaedge AS metaedge
    """
    with driver.session() as session:
        result = [dict(record) for record in session.run(query)]

    rdd = spark.sparkContext.parallelize(result)
    pairs = rdd.map(lambda x: (x['Compound'], x['metaedge']))
    return pairs

In [5]:
def Sort(pairs):

    disease_types = ['CtD', 'CpD']
    diseases = pairs.filter(lambda x: x[1] in disease_types).map(lambda x: (x[0], 1))
    
    return diseases

In [16]:
def Reduce(diseases):
    disease_counts = diseases.reduceByKey(lambda a, b: a + b)
    count_groups = disease_counts.map(lambda x: (x[1], 1))
    grouped_counts = count_groups.reduceByKey(lambda a, b: a + b)
    desc = grouped_counts.sortBy(lambda x: x[0], ascending=False)
    
    return desc

In [18]:
def MapReduce(driver):
    results = Map(driver)
    sorted = Sort(results)
    reduced = Reduce(sorted)
    return reduced.take(5)

In [19]:
results = MapReduce(driver)
results

[(19, 1), (17, 2), (16, 2), (14, 2), (13, 2)]

In [21]:
for disease, compound in results:
    print(f"Compound Count: {compound}, Disease Count: {disease}")

Compound Count: 1, Disease Count: 19
Compound Count: 2, Disease Count: 17
Compound Count: 2, Disease Count: 16
Compound Count: 2, Disease Count: 14
Compound Count: 2, Disease Count: 13


In [20]:
spark.stop()
driver.close()